# 2026年7月3日后：Daily Niño 3.4 与 A股种植业次交易日收益预测

这是一份便于学习和审核的精简版 Notebook，完整流程只有四步：

1. 下载 2026-07-03 之后的 NOAA OISST 日度文件：优先使用 final，final 尚未生成时使用 preliminary。
2. 从每个文件中截取 Niño 3.4 区域并计算 Daily Niño 3.4 海温异常。
3. 下载申万种植业指数 `801016`，把气候数据映射到真正可预测的下一 A 股交易日。
4. 进行 post-7/3 单因子回归和 expanding-window 一步预测，并与零收益、历史均值基准比较。

## 重要研究口径

- 日期为 `d` 的 OISST 被视为在 `d+1` 收盘后已知，因此用于预测该时点之后的第一个 A 股交易日。
- 例如：9月8日 OISST → 9月9日收盘后获知 → 预测9月10日种植业收益。
- 历史日期优先使用当前 final 文件，因此这是**使用修订后气候数据的回溯性预测分析**，不是严格保存历史数据版本的实时交易回测。
- 因变量是目标交易日的 close-to-close 收益，即目标日收盘价相对上一交易日收盘价的变化。

本 Notebook 交付时没有执行任何下载或模型，所有代码单元均为空输出。

In [ ]:
# ============================================================
# 1. 导入工具并设置研究参数
# ============================================================

from __future__ import annotations

import io
from pathlib import Path

import akshare as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
import xarray as xr
from scipy.stats import norm


# 研究起点：只下载和分析 2026年7月3日之后的数据。
START_DATE = pd.Timestamp("2026-07-03")

# OISST 通常至少延迟一天，因此最多请求到昨天。
END_DATE = pd.Timestamp.now(tz="Asia/Shanghai").tz_localize(None).normalize() - pd.Timedelta(days=1)

# 申万种植业指数代码。
PLANTING_SYMBOL = "801016"

# 递归预测优先使用20个交易日作为初始训练窗口。
PREFERRED_INITIAL_WINDOW = 20
MINIMUM_INITIAL_WINDOW = 15

# 结果统一保存到桌面上的独立文件夹。
# 输出写入当前研究子目录下的 results 文件夹。
# 从 GitHub 克隆仓库后，只要在本 Notebook 所在目录启动并运行即可，
# 不再依赖某一台电脑上的绝对路径。
OUTPUT_DIR = Path.cwd() / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("研究日期：", START_DATE.date(), "至", END_DATE.date())
print("结果目录：", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. 构造 NOAA final 和 preliminary 下载地址
# ============================================================

def build_oisst_urls(day: pd.Timestamp) -> tuple[str, str]:
    """返回某一天 OISST 的 final 地址和 preliminary 地址。"""
    month = day.strftime("%Y%m")
    stamp = day.strftime("%Y%m%d")

    # final 文件一般约两周后生成，文件名不带 preliminary。
    final = (
        "https://www.ncei.noaa.gov/data/"
        "sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr/"
        f"{month}/oisst-avhrr-v02r01.{stamp}.nc"
    )

    # preliminary 是 NOAA 提供的近期近实时版本。
    preliminary = (
        "https://www.ncei.noaa.gov/thredds/fileServer/"
        "ncFC/fc-oisst-daily-avhrr-only-dly-prelim/files/"
        f"{month}/oisst-avhrr-v02r01.{stamp}_preliminary.nc"
    )
    return final, preliminary


def download_one_day(day: pd.Timestamp) -> tuple[bytes, str, str] | None:
    """
    下载某一天的 OISST。

    下载顺序：
    1. 先尝试 final；
    2. final 不存在（HTTP 404）时再尝试 preliminary；
    3. 两个版本都不存在时返回 None。

    返回值包括：文件内容、数据版本、实际使用的网址。
    """
    final_url, preliminary_url = build_oisst_urls(day)

    for version, url in (("final", final_url), ("preliminary", preliminary_url)):
        response = requests.get(url, timeout=30)

        if response.status_code == 404:
            continue

        response.raise_for_status()

        # 正常单日 OISST 文件远大于100KB；这个检查可避免把错误网页当成NetCDF。
        if len(response.content) < 100_000:
            raise ValueError(f"{day.date()} 下载内容异常小，可能不是有效NetCDF文件")

        return response.content, version, url

    return None

In [ ]:
# ============================================================
# 3. 从单日 OISST 文件计算 Daily Niño 3.4
# ============================================================

def extract_daily_nino34(file_content: bytes) -> tuple[pd.Timestamp, float]:
    """
    从单日 NetCDF 文件提取 Niño 3.4 区域的海温异常平均值。

    Niño 3.4 区域：
    - 纬度：5°S 到 5°N；
    - 经度：170°W 到 120°W；
    - OISST 经度采用0到360，因此经度转换为190到240。

    规则经纬网格在高纬度的网格面积较小，所以使用 cos(latitude)
    作为面积权重。Niño 3.4 靠近赤道，加权与普通平均差异很小，
    但面积加权在方法上更规范。
    """
    with xr.open_dataset(io.BytesIO(file_content), engine="h5netcdf") as dataset:
        if "anom" not in dataset:
            raise ValueError("OISST 文件中没有 anom 海温异常变量")

        region = dataset["anom"].sel(
            lat=slice(-5, 5),
            lon=slice(190, 240),
        ).squeeze(drop=True)

        latitude_weights = np.cos(np.deg2rad(region["lat"]))
        daily_value = region.weighted(latitude_weights).mean(
            dim=("lat", "lon"),
            skipna=True,
        )

        factor_date = pd.Timestamp(dataset["time"].values[0]).normalize()
        value = float(daily_value.item())

    if not np.isfinite(value):
        raise ValueError(f"{factor_date.date()} 的 Niño 3.4 数值无效")

    return factor_date, value

In [ ]:
# ============================================================
# 4. 批量下载7月3日后的 OISST，并生成 Daily Niño 3.4 CSV
# ============================================================

records = []
failed_dates = []

for requested_date in pd.date_range(START_DATE, END_DATE, freq="D"):
    try:
        downloaded = download_one_day(requested_date)

        if downloaded is None:
            failed_dates.append(requested_date)
            print(f"[暂无文件] {requested_date.date()}")
            continue

        file_content, data_version, source_url = downloaded
        factor_date, nino34_value = extract_daily_nino34(file_content)

        # 防止服务器文件日期与请求日期不一致。
        if factor_date != requested_date.normalize():
            raise ValueError(
                f"请求日期为 {requested_date.date()}，但文件日期为 {factor_date.date()}"
            )

        records.append({
            "Factor_Date": factor_date,
            "Nino34_Daily": nino34_value,
            "Data_Version": data_version,
            "Source_URL": source_url,
        })

        print(
            f"[成功] {factor_date.date()}  "
            f"Nino34={nino34_value:.4f}  版本={data_version}"
        )

    except Exception as error:
        failed_dates.append(requested_date)
        print(f"[失败] {requested_date.date()}：{type(error).__name__}: {error}")


nino34_daily = pd.DataFrame(records)

if nino34_daily.empty:
    raise RuntimeError("没有成功下载任何 OISST 文件")

nino34_daily = (
    nino34_daily
    .drop_duplicates("Factor_Date", keep="last")
    .sort_values("Factor_Date")
    .reset_index(drop=True)
)

nino_csv = OUTPUT_DIR / "nino34_daily_post_20260703.csv"
nino34_daily.to_csv(nino_csv, index=False)

print("\nDaily Niño 3.4 数据：")
print(nino34_daily.head())
print(nino34_daily.tail())
print("成功天数：", len(nino34_daily))
print("未取得文件的天数：", len(failed_dates))
print("保存位置：", nino_csv)

In [ ]:
# ============================================================
# 5. 下载申万种植业指数801016并计算日收益率
# ============================================================

# AKShare 返回申万指数的每日价格数据。
planting_raw = ak.index_hist_sw(
    symbol=PLANTING_SYMBOL,
    period="day",
)

required_columns = {"日期", "收盘"}
missing_columns = required_columns.difference(planting_raw.columns)
if missing_columns:
    raise ValueError(f"种植业数据缺少必要列：{sorted(missing_columns)}")

planting = planting_raw[["日期", "收盘"]].copy()
planting = planting.rename(columns={
    "日期": "Target_Trade_Date",
    "收盘": "Planting_Close",
})

planting["Target_Trade_Date"] = pd.to_datetime(
    planting["Target_Trade_Date"], errors="coerce"
).dt.normalize()
planting["Planting_Close"] = pd.to_numeric(
    planting["Planting_Close"], errors="coerce"
)

planting = (
    planting
    .dropna(subset=["Target_Trade_Date", "Planting_Close"])
    .drop_duplicates("Target_Trade_Date", keep="last")
    .sort_values("Target_Trade_Date")
    .reset_index(drop=True)
)

# 当日收益 = 当日收盘价 / 上一交易日收盘价 - 1。
# 因为 planting 只包含交易日，所以 pct_change 自动使用上一交易日，
# 不会把周末或节假日误当成交易日。
planting["R_Plant_1D"] = planting["Planting_Close"].pct_change()

print(planting.tail())

In [ ]:
# ============================================================
# 6. 将每个 Niño 3.4 数值映射到真正可预测的下一交易日
# ============================================================

# 研究假设：Factor_Date 为 d 的数据，在 d+1 的A股收盘后获得。
# 因此它不能预测 d+1 当天，只能预测 d+1 之后的第一个A股交易日。
nino34_daily["Known_After_Date"] = (
    nino34_daily["Factor_Date"] + pd.Timedelta(days=1)
)

trading_dates = planting["Target_Trade_Date"].sort_values().to_numpy(dtype="datetime64[ns]")


def find_next_trading_day(known_after_date: pd.Timestamp) -> pd.Timestamp | pd.NaT:
    """寻找严格晚于数据获知日期的第一个A股交易日。"""
    position = np.searchsorted(
        trading_dates,
        np.datetime64(known_after_date, "ns"),
        side="right",
    )
    if position >= len(trading_dates):
        return pd.NaT
    return pd.Timestamp(trading_dates[position])


nino34_daily["Target_Trade_Date"] = nino34_daily["Known_After_Date"].map(
    find_next_trading_day
)

# 周末附近可能有多个自然日因子映射到同一个交易日。
# 在目标交易日前，我们应使用当时最新的 Factor_Date，所以保留日期最大的那一条。
signal = (
    nino34_daily
    .dropna(subset=["Target_Trade_Date"])
    .sort_values(["Target_Trade_Date", "Factor_Date"])
    .drop_duplicates("Target_Trade_Date", keep="last")
)

model_data = pd.merge(
    signal,
    planting[["Target_Trade_Date", "Planting_Close", "R_Plant_1D"]],
    on="Target_Trade_Date",
    how="inner",
)

model_data = model_data[
    model_data["Target_Trade_Date"] >= START_DATE
].dropna(subset=["Nino34_Daily", "R_Plant_1D"])

model_data = model_data.sort_values("Target_Trade_Date").reset_index(drop=True)

aligned_csv = OUTPUT_DIR / "nino34_planting_aligned.csv"
model_data.to_csv(aligned_csv, index=False)

print(model_data[[
    "Factor_Date", "Known_After_Date", "Target_Trade_Date",
    "Nino34_Daily", "Data_Version", "R_Plant_1D",
]].tail(15))
print("有效交易日样本：", len(model_data))

In [ ]:
# ============================================================
# 7. Post-7/3 全样本预测回归：OLS + HAC标准误
# ============================================================

# 回归模型：
# R_Plant,target = alpha + beta × Nino34_Daily + error
#
# 系数仍由普通最小二乘法估计；HAC/Newey-West只调整标准误，
# 用于降低日频残差异方差和序列相关对显著性判断的影响。

X_full = sm.add_constant(
    model_data[["Nino34_Daily"]],
    has_constant="add",
)
y_full = model_data["R_Plant_1D"]

ols_model = sm.OLS(y_full, X_full).fit()
hac_model = ols_model.get_robustcov_results(
    cov_type="HAC",
    maxlags=5,
)

regression_rows = []

for standard_error_type, fitted_model in (
    ("OLS", ols_model),
    ("HAC(5)", hac_model),
):
    # get_robustcov_results 返回数组，因此按 const、Nino34_Daily 的顺序读取。
    parameters = np.asarray(fitted_model.params)
    standard_errors = np.asarray(fitted_model.bse)
    t_values = np.asarray(fitted_model.tvalues)
    p_values = np.asarray(fitted_model.pvalues)

    for index, variable in enumerate(("alpha", "beta")):
        regression_rows.append({
            "Standard_Error_Type": standard_error_type,
            "Variable": variable,
            "Coefficient": parameters[index],
            "Standard_Error": standard_errors[index],
            "t_stat": t_values[index],
            "p_value": p_values[index],
            "R_squared": ols_model.rsquared,
            "N": int(ols_model.nobs),
        })

regression_results = pd.DataFrame(regression_rows)
regression_results.to_csv(
    OUTPUT_DIR / "post_break_full_sample_regression.csv",
    index=False,
)

print(regression_results.to_string(index=False))

In [ ]:
# ============================================================
# 8. Expanding-window 一步预测
# ============================================================

def select_initial_window(sample_size: int) -> int:
    """
    优先使用20个交易日训练；如果样本较短，则使用15日。
    至少保留5个真正的样本外预测，否则停止。
    """
    if sample_size - PREFERRED_INITIAL_WINDOW >= 5:
        return PREFERRED_INITIAL_WINDOW
    if sample_size - MINIMUM_INITIAL_WINDOW >= 5:
        return MINIMUM_INITIAL_WINDOW
    raise ValueError(
        f"有效样本只有 {sample_size}，不足以完成15日初始窗口和至少5次预测"
    )


initial_window = select_initial_window(len(model_data))
forecast_rows = []

for test_position in range(initial_window, len(model_data)):
    # 每次训练只使用当前预测日之前已经发生的交易日。
    train = model_data.iloc[:test_position].copy()
    test = model_data.iloc[test_position]

    X_train = sm.add_constant(
        train[["Nino34_Daily"]],
        has_constant="add",
    )
    y_train = train["R_Plant_1D"]
    recursive_model = sm.OLS(y_train, X_train).fit()

    X_test = pd.DataFrame({
        "const": [1.0],
        "Nino34_Daily": [test["Nino34_Daily"]],
    })

    enso_prediction = float(recursive_model.predict(X_test).iloc[0])

    forecast_rows.append({
        "Factor_Date": test["Factor_Date"],
        "Known_After_Date": test["Known_After_Date"],
        "Target_Trade_Date": test["Target_Trade_Date"],
        "Data_Version": test["Data_Version"],
        "Actual_Return": test["R_Plant_1D"],
        "Pred_ENSO_Model": enso_prediction,
        "Pred_Zero_Return": 0.0,
        # 历史均值也必须只使用预测形成前的训练样本。
        "Pred_Expanding_Mean": float(y_train.mean()),
        "Training_N": len(train),
    })

predictions = pd.DataFrame(forecast_rows)
predictions.to_csv(
    OUTPUT_DIR / "post_break_recursive_predictions.csv",
    index=False,
)

print("初始训练窗口：", initial_window)
print("样本外预测次数：", len(predictions))
print(predictions.tail())

In [ ]:
# ============================================================
# 9. 计算 RMSE、MAE 和方向准确率
# ============================================================

actual = predictions["Actual_Return"].to_numpy()

metric_rows = []

for model_name, prediction_column in (
    ("ENSO_Model", "Pred_ENSO_Model"),
    ("Zero_Return", "Pred_Zero_Return"),
    ("Expanding_Historical_Mean", "Pred_Expanding_Mean"),
):
    predicted = predictions[prediction_column].to_numpy()
    errors = actual - predicted

    rmse = float(np.sqrt(np.mean(errors ** 2)))
    mae = float(np.mean(np.abs(errors)))

    # 零收益基准始终预测0，没有上涨或下跌方向，因此方向准确率不适用。
    if model_name == "Zero_Return":
        directional_accuracy = np.nan
    else:
        directional_accuracy = float(
            np.mean(np.sign(predicted) == np.sign(actual))
        )

    metric_rows.append({
        "Model": model_name,
        "N": len(actual),
        "RMSE": rmse,
        "MAE": mae,
        "Directional_Accuracy": directional_accuracy,
    })

metrics = pd.DataFrame(metric_rows)

# OOS R² 的定义：1 - ENSO模型MSE / 基准模型MSE。
# 正数表示 ENSO 模型的样本外均方误差更低；负数表示表现更差。
enso_mse = metrics.loc[metrics["Model"] == "ENSO_Model", "RMSE"].iloc[0] ** 2

for benchmark in ("Zero_Return", "Expanding_Historical_Mean"):
    benchmark_mse = metrics.loc[metrics["Model"] == benchmark, "RMSE"].iloc[0] ** 2
    metrics.loc[
        metrics["Model"] == "ENSO_Model",
        f"OOS_R2_vs_{benchmark}",
    ] = 1 - enso_mse / benchmark_mse

metrics.to_csv(
    OUTPUT_DIR / "post_break_forecast_metrics.csv",
    index=False,
)

print(metrics.to_string(index=False))

In [ ]:
# ============================================================
# 10. 检查 ENSO 是否真的改善了涨跌方向判断
# ============================================================

# 方向准确率相同不代表两个模型每天的预测必然完全相同。
# 因此这里逐日比较：哪些日期只有 ENSO 正确，哪些日期只有历史均值正确。
direction_comparison = predictions[[
    "Target_Trade_Date", "Actual_Return",
    "Pred_ENSO_Model", "Pred_Expanding_Mean",
]].copy()

direction_comparison["Actual_Direction"] = np.sign(direction_comparison["Actual_Return"])
direction_comparison["ENSO_Direction"] = np.sign(direction_comparison["Pred_ENSO_Model"])
direction_comparison["Mean_Direction"] = np.sign(direction_comparison["Pred_Expanding_Mean"])

direction_comparison["ENSO_Correct"] = (
    direction_comparison["ENSO_Direction"] == direction_comparison["Actual_Direction"]
)
direction_comparison["Mean_Correct"] = (
    direction_comparison["Mean_Direction"] == direction_comparison["Actual_Direction"]
)

direction_agreement = float(
    (direction_comparison["ENSO_Direction"] == direction_comparison["Mean_Direction"]).mean()
)
enso_only_correct = int(
    (direction_comparison["ENSO_Correct"] & ~direction_comparison["Mean_Correct"]).sum()
)
mean_only_correct = int(
    (~direction_comparison["ENSO_Correct"] & direction_comparison["Mean_Correct"]).sum()
)

direction_summary = pd.DataFrame([{
    "N": len(direction_comparison),
    "Direction_Agreement_Rate": direction_agreement,
    "ENSO_Only_Correct": enso_only_correct,
    "Historical_Mean_Only_Correct": mean_only_correct,
    "Net_Extra_Correct_From_ENSO": enso_only_correct - mean_only_correct,
}])

direction_comparison.to_csv(OUTPUT_DIR / "direction_comparison_by_day.csv", index=False)
direction_summary.to_csv(OUTPUT_DIR / "direction_comparison_summary.csv", index=False)

print(direction_summary.to_string(index=False))
print("\n正确性配对表：")
print(pd.crosstab(
    direction_comparison["ENSO_Correct"],
    direction_comparison["Mean_Correct"],
    rownames=["ENSO正确"],
    colnames=["历史均值正确"],
))

In [ ]:
# ============================================================
# 11. 预测误差比较：DM 与 Clark–West
# ============================================================

# OOS R² 是表现指标，不提供显著性。
# 这里另外比较每天的平方预测误差：
# - DM/HAC：ENSO 与两个基准的误差是否有系统差异；
# - Clark–West：针对“预测回归 vs 历史均值”这种嵌套模型的单侧检验。

actual_values = predictions["Actual_Return"].to_numpy()
enso_forecasts = predictions["Pred_ENSO_Model"].to_numpy()
enso_errors = actual_values - enso_forecasts

test_rows = []

for benchmark_name, benchmark_column in (
    ("Zero_Return", "Pred_Zero_Return"),
    ("Expanding_Historical_Mean", "Pred_Expanding_Mean"),
):
    benchmark_forecasts = predictions[benchmark_column].to_numpy()
    benchmark_errors = actual_values - benchmark_forecasts

    # 正的 loss_gain 表示 ENSO 模型的平方误差更小。
    loss_gain = benchmark_errors ** 2 - enso_errors ** 2
    dm_model = sm.OLS(loss_gain, np.ones((len(loss_gain), 1))).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 1},
    )

    row = {
        "Benchmark": benchmark_name,
        "N": len(loss_gain),
        "Mean_MSE_Gain": float(loss_gain.mean()),
        "DM_HAC1_t": float(dm_model.tvalues[0]),
        "DM_HAC1_p_two_sided": float(dm_model.pvalues[0]),
    }

    if benchmark_name == "Expanding_Historical_Mean":
        # Clark–West 调整项用于预测回归相对历史均值的比较。
        cw_adjusted = loss_gain + (enso_forecasts - benchmark_forecasts) ** 2
        cw_model = sm.OLS(cw_adjusted, np.ones((len(cw_adjusted), 1))).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": 1},
        )
        row["Clark_West_t"] = float(cw_model.tvalues[0])
        row["Clark_West_p_one_sided"] = float(1 - norm.cdf(cw_model.tvalues[0]))

    test_rows.append(row)

forecast_tests = pd.DataFrame(test_rows)
forecast_tests.to_csv(OUTPUT_DIR / "forecast_comparison_tests.csv", index=False)
print(forecast_tests.to_string(index=False))

In [ ]:
# ============================================================
# 12. 绘制实际收益与 ENSO 模型预测
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    predictions["Target_Trade_Date"],
    predictions["Actual_Return"] * 100,
    marker="o",
    # 图中文字使用英文，避免不同电脑缺少中文字体时出现方框或警告；
    # Notebook 的解释和代码注释仍保留中文。
    label="Actual return",
)

ax.plot(
    predictions["Target_Trade_Date"],
    predictions["Pred_ENSO_Model"] * 100,
    marker="o",
    label="ENSO model forecast",
)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Post-2026-07-03 ENSO recursive one-step-ahead forecast")
ax.set_xlabel("Target trading date")
ax.set_ylabel("Planting-sector daily return (%)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()

figure_path = OUTPUT_DIR / "post_break_recursive_forecast.png"
fig.savefig(figure_path, dpi=180)
plt.show()

print("图表保存位置：", figure_path)
print("\n全部输出文件：")
for output_file in sorted(OUTPUT_DIR.glob("*")):
    print(output_file)

## 如何解释结果

### 全样本回归

- `beta`：Daily Niño 3.4 每增加 1°C，目标交易日种植业收益的平均变化。
- `p_value`：用于判断样本中 beta 是否显著偏离0。
- `R_squared`：单一 ENSO 因子解释了多少收益波动。
- 因 post-7/3 样本很短，即使显著，也只能表述为短窗口探索性证据。

### 样本外预测

- `RMSE`、`MAE` 越低越好。
- `Directional_Accuracy` 表示预测涨跌方向与实际方向一致的比例。
- ENSO 模型应同时与零收益和递归历史均值比较，不能只看自身误差。

### 数据版本限制

本 Notebook 对历史日期优先使用当前 final 文件，并在最近日期使用 preliminary。因此结果适合研究“使用当前最佳 OISST 序列时是否存在预测关系”，但不能声称完全复原了每个历史交易日当时真实看到的 preliminary 数值。